# Drought raster indicators — Copernicus EDO / GDO

The `earthlens.drought` backend is **mixed-output**: the US Drought Monitor route
returns vector polygons (see the [quickstart](quickstart.ipynb)), while the
**Copernicus European / Global Drought Observatory** route returns **raster**
GeoTIFFs. This notebook teaches that raster route end-to-end — download an
indicator for a date and bounding box, open the GeoTIFF, and map it.

EDO/GDO is a Copernicus REST *shim*, not a conformant OGC WCS server: only its
`GetCoverage` operation is reliable. The backend builds the documented
`GetCoverage` URL by hand (`TIME` + `SELECTED_TIMESCALE` + a `SUBSET=Long/Lat`
bbox), streams the GeoTIFF, and opens it through
[`pyramids`](https://github.com/serapeum-org/pyramids) — no `owslib`, no WCS reader
WCS driver. Because the resolved dataset is raster, `download()` returns a
`list[Path]` of written GeoTIFFs (one per requested period), not an in-memory
collection.

## Setup

Imports and a scratch output directory. The raster routes write GeoTIFFs to `path/`, so every raster request needs an explicit `path=`.

In [ ]:
import tempfile
from pathlib import Path

from pyramids.dataset import Dataset

from earthlens.core import EarthLens
from earthlens.drought import Catalog

OUT = Path(tempfile.mkdtemp(prefix="edo-"))
OUT

## Quickstart — one EDO indicator, one date, one bbox

`edo-spaST` is the **SPI (ERA5) short-term** indicator — a standardised
precipitation index. We request a single date over a small European box. The
facade routes the `drought` key to the backend; `dataset=` picks the catalog
row, `lat_lim` / `lon_lim` are the bounding box, and `path=` is where the
GeoTIFF lands.

| argument | meaning | value here |
|----------|---------|------------|
| `dataset` | catalog row id | `edo-spaST` (SPI ERA5 short-term) |
| `start` / `end` | inclusive date window | a single day, `2025-12-21` |
| `lat_lim` / `lon_lim` | bounding box (degrees) | central Europe |
| `path` | output directory (required for raster) | the scratch dir |

In [ ]:
paths = EarthLens(
    data_source="drought",
    dataset="edo-spaST",
    variables=[],
    start="2025-12-21",
    end="2025-12-21",
    lat_lim=[40.0, 50.0],
    lon_lim=[5.0, 15.0],
    path=str(OUT),
).download(progress_bar=False)

paths

`download()` returned a `list[Path]` — one GeoTIFF for the single requested date. That list-of-paths shape is what every raster drought dataset returns.

## Inspect the GeoTIFF

Open the written raster with `pyramids.dataset.Dataset`. It reports the CRS
(EPSG:4326, as the catalog row declares), the pixel grid, and the geographic
bounds — which should match the box we asked for.

In [ ]:
ds = Dataset.read_file(paths[0])
print("CRS EPSG:", ds.epsg)
print("grid (rows x cols):", ds.rows, "x", ds.columns)
print("bounds [x_min, y_min, x_max, y_max]:", ds.bbox)

## Map the indicator

Read the single band into a NumPy array, mask the no-data fill, and draw it with
its geographic extent so the axes are longitude / latitude. Diverging colours
suit a standardised index (negative = drier than normal, positive = wetter).

In [ ]:
glyph = ds.plot(cmap="RdYlBu", title="EDO SPI short-term — 2025-12-21")
glyph.cbar.set_label("SPI (ERA5, short-term)")

## GDO is the same route, on global data

The European (EDO) and Global (GDO) observatories share a **single** Copernicus
WCS map (`map=DO_WCS`); the coverages carry global data, so the "European vs
global" split is purely the bounding box you pass. `gdo-smand` is the ensemble
soil-moisture anomaly. Here we request it over the Horn of Africa — same call
shape, same `list[Path]` result, a different corner of the same global grid.

In [ ]:
gdo_paths = EarthLens(
    data_source="drought",
    dataset="gdo-smand",
    variables=[],
    start="2024-06-21",
    end="2024-06-21",
    lat_lim=[-5.0, 15.0],
    lon_lim=[30.0, 52.0],
    path=str(OUT),
).download(progress_bar=False)

gdo = Dataset.read_file(gdo_paths[0])

glyph = gdo.plot(
    cmap="BrBG",
    title="GDO ensemble soil-moisture anomaly — Horn of Africa, 2024-06-21",
)
glyph.cbar.set_label("soil-moisture anomaly")

## Discover the available indicators

The catalog carries every EDO and GDO indicator as its own row. Each raster row
declares `transport: edo-wcs`, its Copernicus `coverage` id, a `cadence`, and
the `timescale` (the `SELECTED_TIMESCALE` window). Browse them straight off the
catalog.

In [ ]:
cat = Catalog()
edo = sorted(i for i in cat.datasets if i.startswith("edo-"))
gdo = sorted(i for i in cat.datasets if i.startswith("gdo-"))
print(f"{len(edo)} EDO + {len(gdo)} GDO indicators")

row = cat.get("edo-spaST")
row.transport, row.coverage, row.cadence, row.timescale

## Takeaway

- The Copernicus **EDO / GDO** route is the drought backend's **raster** face:
  `download()` returns a `list[Path]` of GeoTIFFs, one per requested period.
- The backend hits Copernicus' `GetCoverage` REST endpoint directly — no WCS
  handshake — and opens the bytes with `pyramids`.
- EDO and GDO ride one global `DO_WCS` map; the bounding box you pass decides
  which region you get.
- Open any output with `pyramids.dataset.Dataset` to inspect or map it.

Next: the [catalog explorer](catalog_explorer.ipynb) for the full dataset list,
or the [quickstart](quickstart.ipynb) for the USDM vector route.